In [ ]:
import os
import sys
sys.path.append('./coeqwalpackage')

import pandas as pd
import matplotlib.pyplot as plt

from coeqwalpackage.metrics import percent_change_from_baseline
from coeqwalpackage import cqwlutils as cu
from coeqwalpackage.plotting import (custom_parallel_coordinates_highlight_scenarios, custom_parallel_coordinates_highlight_scenarios_baseline_at_zero, plot_tier_radar)

In [ ]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'
   
(ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, *_) = cu.read_init_file(CtrlFile, CtrlTab)

performance_metrics_base = os.path.join(ScenarioDir, "Performance_Metrics")
tiers_base = os.path.join(performance_metrics_base, "Tiered_Outcome_Measures")
metrics_base = os.path.join(performance_metrics_base, "Metrics")
plots_base = os.path.join(performance_metrics_base, "Plots")

os.makedirs(performance_metrics_base, exist_ok=True)
os.makedirs(metrics_base, exist_ok=True)
os.makedirs(tiers_base, exist_ok=True)
os.makedirs(plots_base, exist_ok=True)

# metrics_path = os.path.join(GroupDataDirPath, "metrics_output", "all_metrics_output.csv")
metrics_path = os.path.join(ScenarioDir, "Performance_Metrics", "Metrics", "All_Metrics", "all_metrics_output.csv")

df = pd.read_csv(metrics_path, index_col=0)
df.index.name = 'Scenario'

flood_all = pd.read_csv(os.path.join(metrics_base, "Reservoir_FloodRisk", "floodrisk_all_metrics.csv"), index_col=0)
storage_all = pd.read_csv(os.path.join(metrics_base, "Reservoir_Storage", "storage_all_metrics.csv"), index_col=0)
sal_all = pd.read_csv(os.path.join(metrics_base, "Salinity", "salinity_all_metrics.csv"), index_col=0)
gw_all = pd.read_csv(os.path.join(metrics_base, "Groundwater", "groundwater_all_metrics.csv"), index_col=0)

flood_tiers = flood_all[[c for c in flood_all.columns if 'FloodTier' in c]]
storage_tiers = storage_all[[c for c in storage_all.columns if '_Tier' in c]]
sal_tiers = sal_all[[c for c in sal_all.columns if 'Tier' in c]]
gw_tiers = gw_all[[c for c in gw_all.columns if '_Tier' in c]]
tiers_df = pd.concat([flood_tiers, storage_tiers, sal_tiers, gw_tiers], axis=1, join='outer').sort_index()

### Scenario Groupings

In [ ]:
# Proposed groupings for comparison and review:

# Representing current operations for California water  - all the “baseline” or reference scenarios together; showing the variability in representation of our recent reality
# S0020 - Baseline 2023 DCR operations (same as s0011) with updated 2020 ag land use and TUCP actions
# S0011 - DCR2023 adjusted historical, current ops withTUCP actions are enabled
# S0021 - Baseline 2023 DCR operations without TCUP actions and 2020 land use (same as s0020 but without TUCP actions; also same as s0002 but with updated ag land use)
# S0024 - Baseline -Same as s0023 but with TUCPs active (analogous to USBRs Alt2V1 with TUCPs but with DWR's adj historical hydroclimate and updated land use)
# S0023 - Baseline - USBR's 2024 LTO alternative Alt2V1 (adjusted operations for CVP) combined with DWR's adjusted historical hydrology with updated 2020 LandIQ land use; no TUCPs
# S0047 - Same as s0020 but using DWR's 2023 DCR climate change "50% level of concern" future climate condition
# S0051 - Same as s0023 but using DWR's 2023 DCR climate change "50% level of concern" future climate condition

# Managing groundwater in a changing agricultural landscape  - actions to improve regional declining groundwater conditions
# Baseline/reference scenario: s0020
# S0025 - (SJV pumping limits)
# S0026 - (SJV ag acreage reductions)
# S0027 - (CV-wide pumping limits)
# S0028 - (CV-wide ag acreage reductions)
# S0062 - Same as s0027 but using DWR's 2023 DCR climate change "50% level of concern" climate condition
# S0063 - Same as s0027 but using DWR's 2023 DCR climate change "95% level of concern" climate condition

# Enhancing river flows for the environment  - modifying flow requirements on Bay-Delta tributaries for ecological and environmental purposes and exploring system sensitivities
# Baseline/reference scenario: s0020
# S0030 - Removing existing flow requirements
# S0029 - functional flows on 17 tributary locations
# S0032 - functional flows on 17 tributary locations with ag demand reduction
# S0031 - functional flows for salmon with Shasta cold pool protection
# S0033 - functional flows for salmon with ag demand reduction
# S0046 - Same as s0029 except that the downstream requirements on the Sac and SJR and Delta are de-activated so that it is only tributary and upper reaches.

# Improving flows for the health of the Bay Delta ecosystem 
# Baseline/reference scenario: s0020
# S0040 - Alt3 with 35% unimpaired Delta outflow
# S0041 - Alt3 with 45% unimpaired Delta outflow
# S0042 - Alt3 with 55% unimpaired Delta outflow
# S0039 - Alt3 with 65% unimpaired Delta outflow

# Sustaining uses in the Delta for communities and farms
# Baseline/reference scenario: s0020
# S0042 - Alt3 with 55% unimpaired Delta outflow
# S0039 - Alt3 with 65% unimpaired Delta outflow
# S0044 - increase Shasta carryover target by 20%
# S0045 - remove fall X2
# S0028 - (CV-wide ag acreage reductions)

# Improving reliability of Delta exports for farms and cities
# Baseline/reference scenario: s0020
# S0040 - Alt3 with 35% unimpaired Delta outflow
# S0030 - Removing existing flow requirements
# S0028 - (CV-wide ag acreage reductions)
# S0065 - Delta conveyance project (DCP) - updated version

# Prioritizing community water systems (NOT YET AVAILABLE)
# Baseline/reference scenario: s0020
# S0035
# S0036
# s0037

In [ ]:
SCENARIO_SETS = [
    {"name": "s20_s11_s21_s23_s24_s47_s51", "baseline": 20, "compare": [11, 21, 24, 23, 47, 51]},
    {"name": "s20_s25_s26_s27_s28_s62_s63", "baseline": 20, "compare": [25, 26, 27, 28, 62, 63]},
    {"name": "s20_s29_s30_s31_s32_s33_s46", "baseline": 20, "compare": [30, 29, 32, 31, 33, 46]},
    {"name": "s20_s39_s41_s42", "baseline": 20, "compare": [40, 41, 42, 39]},
    {"name": "s20_s28_s39_s42_s44_s45", "baseline": 20, "compare": [42, 39, 44, 45, 28]},
    {"name": "s20_s28_s30_s40_s65", "baseline": 20, "compare": [40, 30, 28, 65]}
    # {"name": "s20_s35_s36_s37", "baseline": 20, "compare": [35, 36, 37]} # NOT YET AVAILABLE
]

### Definitions

In [ ]:
FIGSIZE = (20, 7)
COLORS = ['black', 'red', 'blue', 'green', 'orange', 'purple', 'yellow', 'cyan', 'magenta']

# Radar plot mode flags
PLOT_KDE = False
PLOT_HISTOGRAM = False
PLOT_OVERLAY = True

for s in SCENARIO_SETS:
    s["baseline_id"] = f's{s["baseline"]:04d}'
    s["compare_ids"] = [f's{c:04d}' for c in s["compare"]]
    s["all_ids"] = [s["baseline_id"]] + s["compare_ids"]
    s["colors"] = COLORS[:len(s["all_ids"])]
    s["labels"] = [f'{s["baseline_id"]} (Baseline)'] + s["compare_ids"]

METRIC_GROUPS = {
    'Storage_Apr': [c for c in df.columns if c.startswith('Apr') and 'S_' in c and 'TAF' in c],
    'Deliveries': [c for c in df.columns if 'DEL_' in c],
    'Salinity': [c for c in df.columns if 'X2_' in c or 'EC_' in c]
}
METRIC_GROUPS = {k: v for k, v in METRIC_GROUPS.items() if v}

TIER_GROUPS = {
    'Tier_FloodRisk': [c for c in tiers_df.columns if 'FloodTier' in c],
    'Tier_Storage': [c for c in tiers_df.columns if 'Storage_Tier' in c],
    'Tier_Salinity': [c for c in tiers_df.columns if 'Salinity' in c],
    'Tier_GW': [c for c in tiers_df.columns if '_Tier' in c and 'WBA' in c]}
TIER_GROUPS = {k: v for k, v in TIER_GROUPS.items() if v}

def short_label(c):
    for prefix in ['Apr_Avg_', 'Sep_Avg_', 'Ann_Avg_', 'Fall_Ann_Avg_', 'Spring_Ann_Avg_', 'GW_', 'S_']:
        if c.startswith(prefix):
            c = c[len(prefix):]
    for suffix in ['_TAF', '_CFS', '_KM', '_UMHOS/CM', '_FloodTier', '_Storage_Tier', '_Tier']:
        c = c.replace(suffix, '')
    return c.rstrip('_')

print("Metric Groups:")
for k, v in METRIC_GROUPS.items():
    print(f"  {k}: {len(v)} cols")

print("\nTier Groups:")
for k, v in TIER_GROUPS.items():
    print(f"  {k}: {len(v)} cols")

## 1. Parallel Line Plots

In [ ]:
for sset in SCENARIO_SETS:    
    for grp, cols in METRIC_GROUPS.items():
        labels = [short_label(c) for c in cols]
        ideal = 'bottom' if 'Salinity' in grp else 'top'
        minmax = ['min'] * len(cols) if 'Salinity' in grp else ['max'] * len(cols)
        
        save_path = os.path.join(plots_base, f"parallel_{sset['name']}_{grp}.png")
        custom_parallel_coordinates_highlight_scenarios(
            objs=df[cols], columns_axes=cols, axis_labels=labels,
            ideal_direction=ideal, minmaxs=minmax,
            highlight_indices=sset['all_ids'], highlight_colors=sset['colors'],
            highlight_descriptions=sset['labels'],
            title=f"{sset['name']}: {grp}", fontsize=10, figsize=FIGSIZE,
            save_fig_filename=save_path, show_units=True)
        plt.show()
        print(f"Saved: {save_path}")

In [ ]:
for sset in SCENARIO_SETS:
    for grp, cols in METRIC_GROUPS.items():
        if sset['baseline_id'] not in df.index:
            print(f"Baseline {sset['baseline_id']} not found")
            continue
        
        pct_df = percent_change_from_baseline(df[cols], sset['baseline_id'])
        labels = [short_label(c) for c in cols]
        
        save_path = os.path.join(plots_base, f"parallel_pct_{sset['name']}_{grp}.png")
        custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
            objs=pct_df, columns_axes=cols, axis_labels=labels,
            highlight_indices=sset['all_ids'], highlight_colors=sset['colors'],
            highlight_descriptions=sset['labels'],
            title=f"{sset['name']}: {grp} — % Change from {sset['baseline_id']}",
            fontsize=10, figsize=FIGSIZE,
            save_fig_filename=save_path)
        
        plt.show()
        print(f"Saved: {save_path}")

## 2. NOD/SOD Radar and Parallel Line Plots

In [ ]:
NOD_reservoirs = ["S_SHSTA_Storage_Tier", "S_TRNTY_Storage_Tier","S_OROVL_Storage_Tier", "S_FOLSM_Storage_Tier"]

SOD_reservoirs = ["S_MELON_Storage_Tier", "S_MLRTN_Storage_Tier", "S_SLUIS_CVP_Storage_Tier", "S_SLUIS_SWP_Storage_Tier"]

NOD_gw = ["WBA2_Tier", "WBA3_Tier", "WBA4_Tier", "WBA5_Tier", "WBA6_Tier", "WBA7N_Tier", "WBA7S_Tier", "WBA8N_Tier", "WBA8S_Tier", "WBA9_Tier", "WBA10_Tier", "WBA11_Tier", "WBA12_Tier", "WBA13_Tier", "WBA14_Tier", "WBA15N_Tier", "WBA15S_Tier", "WBA16_Tier", "WBA17N_Tier", "WBA17S_Tier", "WBA18_Tier", "WBA19_Tier", "WBA20_Tier", "WBA21_Tier", "WBA22_Tier", "WBA23_Tier", "WBA24_Tier", "WBA25_Tier", "WBA26N_Tier", "WBA26S_Tier"]

SOD_gw = ["WBA50_Tier", "WBA60N_Tier", "WBA60S_Tier", "WBA61_Tier", "WBA62_Tier", "WBA63_Tier", "WBA64_Tier", "WBA71_Tier", "WBA72_Tier", "WBA73_Tier", "WBA90_Tier"]

Salinity_InDelta = ['Salinity_InDelta_Tier']

Salinity_Export = ['Salinity_Export_Tier']

In [ ]:
print(f"GW tiers available: {[c for c in gw_all.columns if '_Tier' in c][:5]}... ({len([c for c in gw_all.columns if '_Tier' in c])} total)")
print(f"Storage tiers available: {[c for c in storage_all.columns if '_Tier' in c]}")
print(f"Salinity tiers available: {[c for c in sal_all.columns if 'Tier' in c]}")

In [ ]:
nod_sod_df = pd.DataFrame(index=gw_all.index)
nod_sod_df.index.name = 'scenario'

nod_sod_df["NOD_GW_Mean"] = gw_all[NOD_gw].mean(axis=1)
nod_sod_df["SOD_GW_Mean"] = gw_all[SOD_gw].mean(axis=1)

nod_sod_df["NOD_Reservoir_Mean"] = storage_all[NOD_reservoirs].mean(axis=1).reindex(nod_sod_df.index)
nod_sod_df["SOD_Reservoir_Mean"] = storage_all[SOD_reservoirs].mean(axis=1).reindex(nod_sod_df.index)

nod_sod_df["Export_Salinity"] = sal_all["Salinity_Export_Tier"].reindex(nod_sod_df.index)
nod_sod_df["InDelta_Salinity"] = sal_all["Salinity_InDelta_Tier"].reindex(nod_sod_df.index)

nod_sod_df.reset_index(inplace=True)
nod_sod_df

In [ ]:
nod_sod_std_df = pd.DataFrame(index=gw_all.index)
nod_sod_std_df.index.name = 'scenario'

if len(NOD_gw) >= 2:
    nod_sod_std_df["NOD_GW_Std"] = gw_all[NOD_gw].std(axis=1)
else:
    nod_sod_std_df["NOD_GW_Std"] = 0
    
if len(SOD_gw) >= 2:
    nod_sod_std_df["SOD_GW_Std"] = gw_all[SOD_gw].std(axis=1)
else:
    nod_sod_std_df["SOD_GW_Std"] = 0
    
if len(NOD_reservoirs) >= 2:
    nod_sod_std_df["NOD_Reservoir_Std"] = storage_all[NOD_reservoirs].std(axis=1).reindex(nod_sod_std_df.index)
else:
    nod_sod_std_df["NOD_Reservoir_Std"] = 0
    
if len(SOD_reservoirs) >= 2:
    nod_sod_std_df["SOD_Reservoir_Std"] = storage_all[SOD_reservoirs].std(axis=1).reindex(nod_sod_std_df.index)
else:
    nod_sod_std_df["SOD_Reservoir_Std"] = 0
    
if len(Salinity_Export) >= 2:
    nod_sod_std_df["Export_Salinity_Std"] = sal_all[Salinity_Export].reindex(nod_sod_df.index)
else:
    nod_sod_std_df["Export_Salinity_Std"] = 0
    
if len(Salinity_InDelta) >= 2:
    nod_sod_std_df["InDelta_Salinity_Std"] = sal_all[Salinity_InDelta].reindex(nod_sod_df.index)
else:
    nod_sod_std_df["InDelta_Salinity_Std"] = 0

nod_sod_std_df.fillna(0, inplace = True)

nod_sod_std_df.reset_index(inplace=True)
nod_sod_std_df

In [ ]:
cols = ["NOD_GW_Mean", "SOD_GW_Mean", "NOD_Reservoir_Mean", "SOD_Reservoir_Mean", "Export_Salinity", "InDelta_Salinity"]
labels = cols
minmax = ["max", "max", "max", "max", "max", "max"]
ideal = "top"

for sset in SCENARIO_SETS:
    highlight_scenarios = sset['all_ids']
    highlight_indices = nod_sod_df.index[nod_sod_df["scenario"].isin(highlight_scenarios)].tolist()
    
    save_path = os.path.join(plots_base, f"parallel_NOD_SOD_{sset['name']}.png")
    
    fig, ax = custom_parallel_coordinates_highlight_scenarios(
        objs=nod_sod_df[cols],
        columns_axes=cols,
        axis_labels=labels,
        ideal_direction=ideal,
        minmaxs=minmax,
        highlight_indices=highlight_indices,
        highlight_colors=sset['colors'],
        highlight_descriptions=sset['labels'],
        title=f"Averaged NOD and SOD Tiers: {sset['name']}",
        fontsize=12,
        figsize=(22, 10),
        save_fig_filename=save_path)
    
    plt.show()
    print(f"Saved: {save_path}")

## 3. Tier-Based Radar Plots

In [ ]:
if PLOT_KDE:
    for sset in SCENARIO_SETS:
        plot_tier_radar(df=nod_sod_df, cols=cols, scenario_col='scenario', highlight_scenarios=sset['all_ids'], highlight_colors=sset['colors'], highlight_labels=sset['labels'], title=f"Tier Radar: {sset['name']}", std_df=nod_sod_std_df, save_path=os.path.join(plots_base, f"tier_radar_kde_{sset['name']}.png"), mode='kde', kde_bw=0.25)

In [ ]:
if PLOT_HISTOGRAM:
    for sset in SCENARIO_SETS:
        plot_tier_radar(df=nod_sod_df, cols=cols, scenario_col='scenario', highlight_scenarios=sset['all_ids'], highlight_colors=sset['colors'], highlight_labels=sset['labels'], title=f"Tier Radar: {sset['name']}", std_df=nod_sod_std_df, save_path=os.path.join(plots_base, f"tier_radar_histogram_{sset['name']}.png"), mode='histogram', bin_width=0.1)

In [ ]:
if PLOT_OVERLAY:
    for sset in SCENARIO_SETS:
        plot_tier_radar(df=nod_sod_df, cols=cols, scenario_col='scenario', highlight_scenarios=sset['all_ids'], highlight_colors=sset['colors'], highlight_labels=sset['labels'], title=f"Tier Radar: {sset['name']}", std_df=nod_sod_std_df, save_path=os.path.join(plots_base, f"tier_radar_overlay_{sset['name']}.png"), mode='overlay', kde_bw=0.25, bin_width=0.1)